# Go/no-go

In a go/no-go task the animal either responds (go) or withholds, and on no-go trials withholding is the correct
answer. This tutorial takes a go/no-go animal through the same pipeline as the [2-AFC tutorial](two_afc.ipynb) and
shows what changes: the task, how trials are scored, which statistics apply and which the library refuses.

The animal is synthetic: eight sessions in which go is correct for stimuli above the boundary, with the laser on
30% of trials, where it makes the animal more willing to go (it moves the curve's midpoint by −0.2). From files,
the change is the config's task block (`type: go_nogo`);
[`go_nogo_minimal.yaml`](../config_examples/go_nogo_minimal.yaml) is a complete example.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from behav_utils import (
                         GoNoGo,
                         Outcome,
                         PsychometricModel,
                         TrialArrays,
                         compute_psychometric_curve,
                         compute_stats,
                         filter_trials,
                         generate_synthetic_animal,
                         plot_psychometric_curve,
                         select_sessions,
)
from behav_utils.stats import PSYCHOMETRIC, SDT

model = PsychometricModel(mu=0.0, sigma=0.3, lapse_low=0.05, lapse_high=0.05, laser={'mu': -0.2})
animal = generate_synthetic_animal(model, n_sessions=8, trials_per_session=300, task=GoNoGo(),
                                   laser_fraction=0.3, rng=1)
sessions = select_sessions(animal, min_trials=0)
off = filter_trials(sessions, trial_type='non_opto')
on = filter_trials(sessions, trial_type='opto')
arrays = TrialArrays.from_sessions(off)
arrays.n_trials, arrays.n_responded

Every non-aborted trial counts as responded. A go is 1, and a blank that is not an abort is a withhold, 0, so it
is scored rather than lost; a go/no-go session has no no-response trials:

In [ ]:
codes, counts = np.unique(animal.sessions[0].trials.outcome, return_counts=True)
{Outcome(code).name: int(count) for code, count in zip(codes, counts)}

## Statistics

The statistics that need only a binary, scored response on a stimulus axis work as in 2-AFC: accuracy, the
psychometric fit and signal detection (`dprime`, `criterion`). Go/no-go adds its own: `go_bias`, the go rate
minus 0.5, and `hit_minus_cr_rate`, the hit rate minus the correct-rejection rate. The
[statistics reference](../stats_reference.md) defines each.

In [ ]:
compute_stats(arrays, ['accuracy', 'go_bias', 'hit_minus_cr_rate', *SDT, *PSYCHOMETRIC]).round(3)

Statistics that assume two active choices are refused, and the error names the go/no-go counterpart where there
is one (see [Tasks, families and capabilities](../concepts/tasks.md)):

In [ ]:
compute_stats(arrays, ['side_bias'])

## The psychometric curve

For go/no-go the psychometric curve is P(go) against the stimulus:

In [ ]:
curve = compute_psychometric_curve(arrays, n_bootstrap=200)
fig, ax = plt.subplots(figsize=(4.5, 3.5))
plot_psychometric_curve(curve, ax=ax, label='laser off')
ax.set_ylabel('P(go)')
ax.legend();

## A laser contrast

The laser was assigned trial by trial within the same sessions, so trials are a valid resampling unit and a
permutation p is allowed (see [Resampling units](../concepts/resampling.md)). The draws are kept small to keep the
notebook quick; with 100 permutations the smallest possible p is 1/101, about 0.01.

In [ ]:
from behav_utils import compute_delta_stat
from behav_utils.plotting import plot_psychometric_comparison

d = compute_delta_stat({'off': off, 'on': on}, ['go_bias', 'mu'], reference='off',
                       n_bootstrap=100, n_permutations=100)
plot_psychometric_comparison(d)
d.contrast('on').table().round(3)

## Where to go next

- [Continuous responses](continuous.ipynb) does the same for a task whose responses are values.
- [Tasks, families and capabilities](../concepts/tasks.md) explains why the library refuses what it does.
- The [statistics reference](../stats_reference.md) lists the statistics each built-in task refuses.